# 04 — Feature Engineering, Selection, and PCA

## Engineered features
| Feature | Formula | Motivation |
|---------|---------|------------|
| `medals_per_million` | medals / population × 10⁶ | Normalises by population so small and large countries are comparable |
| `log_gdp_per_capita` | log1p(gdp_per_capita_usd) | Compresses right-skewed GDP distribution |
| `log_population` | log1p(population) | Same; population spans ~10⁵–10⁹ |
| `log_medals` | log1p(medals) | Used as model target |
| `gold_share` | gold / medals | Quality of medal haul (gold-heavy vs bronze-heavy) |
| `host_flag` | 1 if country hosted that edition | Binary treatment indicator for H3 |
| `income_quartile` | qcut(gdp_per_capita, 4) | Categorical grouping for Kruskal-Wallis |

## Feature selection rationale
Five numeric features are used in PCA: `log_gdp_per_capita`, `log_population`, `medals_per_million`, `gold_share`, `host_flag`. These are the features directly tied to the two research sub-questions (wealth → rate; size → totals; host flag → advantage).

## Imputation
`gold_share` is NaN when `medals == 0`, which cannot occur in the analytical sample (all rows have at least 1 medal by construction). No imputation is applied. Infinite values from division are guarded against with `replace([inf, -inf], nan)` before PCA.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'tmp' / 'matplotlib'))

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer

data = pd.read_csv(ROOT / 'data/processed/olympics_wealth_country_edition.csv')
features = ['log_gdp_per_capita', 'log_population', 'medals_per_million', 'gold_share', 'host_flag']

X = data[features].replace([float('inf'), float('-inf')], float('nan'))
X_imp = SimpleImputer(strategy='median').fit_transform(X)

pcs = PCA().fit(StandardScaler().fit_transform(X_imp))

pca_table = pd.DataFrame({
    'component': [f'PC{i}' for i in range(1, len(features)+1)],
    'explained_variance_ratio': pcs.explained_variance_ratio_,
    'cumulative_variance': pcs.explained_variance_ratio_.cumsum()
})
print('PCA explained variance:')
display(pca_table)

### Scree plot and cumulative variance

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
components = range(1, len(features) + 1)
ax.bar(components, pcs.explained_variance_ratio_, alpha=0.45, color='#60a5fa', label='Individual')
ax.plot(components, pcs.explained_variance_ratio_.cumsum(), marker='o', color='#1d4ed8', label='Cumulative')
ax.axhline(0.80, ls='--', color='#dc2626', lw=1, label='80% threshold')
ax.set_xticks(list(components))
ax.set_xticklabels([f'PC{i}' for i in components])
ax.set_ylim(0, 1.05)
ax.set_xlabel('Principal Component')
ax.set_ylabel('Explained variance ratio')
ax.set_title('PCA scree plot — Olympics × Wealth feature set')
ax.legend()
plt.tight_layout()
plt.show()

### PCA loadings — which features drive each component?

In [ ]:
loadings = pd.DataFrame(
    pcs.components_.T,
    index=features,
    columns=[f'PC{i}' for i in range(1, len(features)+1)]
).round(3)
print('PCA loadings (feature contributions to each PC):')
display(loadings)

### Feature correlation matrix

In [ ]:
corr = pd.DataFrame(X_imp, columns=features).corr().round(3)
print('Pearson correlation matrix:')
display(corr)

# VIF from inverse correlation matrix diagonal
inv = np.linalg.pinv(corr.to_numpy())
vif = pd.DataFrame({'feature': features, 'VIF': np.diag(inv).round(2)})
print('\nVariance Inflation Factors (diagonal of inverse correlation matrix):')
display(vif)
print('\nNote: VIF < 5 → no problematic multicollinearity; VIF > 10 → concern')

### Descriptive overperformers: countries winning more than their GDP/population predict

In [ ]:
# Log-linear model: log(medals) ~ log(gdp_per_cap) + log(pop)
design = np.column_stack([np.ones(len(data)), data.log_gdp_per_capita, data.log_population])
coef, *_ = np.linalg.lstsq(design, data.log_medals, rcond=None)
data['predicted_log_medals'] = design @ coef
data['overperformance_residual'] = data.log_medals - data.predicted_log_medals

top_over = (
    data.sort_values('overperformance_residual', ascending=False)
    .groupby('NOC', as_index=False).first()
    .head(15)
    .sort_values('overperformance_residual', ascending=False)
)
print('Top 15 descriptive overperformers (best edition per NOC):')
display(top_over[['NOC','Year','Season','medals','gdp_per_capita_usd','overperformance_residual']].reset_index(drop=True))